# 1. Project Scope

* **Dataset:** Yambda (Yandex.Music) user–item interaction logs with `uid`, 5-second-binned delta `timestamp`, `item_id`, `event_type` (listen/like/unlike/dislike/undislike), `is_organic`, and listen engagement (`played_ratio_pct`, `track_length_seconds`).

* **Course techniques:**

  * **Graph mining** (popularity, degree distributions, long-tail)
  * **Clustering / segmentation** (users by activity, sessions by behavior)
  * **Anomaly detection** (extreme activity users, unusual repetition, bursts)

* **External techniques**

  * **Sessionization + sequential pattern mining** (within-session sequences, transition probabilities)
  * **Session-based recommenders / next-item prediction** (Markov baselines → RNN/Transformer)
  * Bias-aware modeling using **organic vs non-organic** as context

**EDA findings (key takeaways)**

- In the Yambda 50M interaction log (50M split), activity is dominated by `listen` events (about 46.5M), with smaller but still informative explicit feedback actions (`like`, `unlike`, `dislike`, `undislike`).
- Data quality checks show ~2.77% of rows are missing both engagement fields (`played_ratio_pct` and `track_length_seconds`), about 0.47% of non-missing `played_ratio_pct` values exceed 100 (only for listens), and roughly 1.25% of rows are duplicates under (uid, timestamp, item_id, event_type).
- The user–item space is extremely sparse and long-tailed: with ~934k items, the median item has only 3 interactions and 73.7% of items have ≤10 interactions, while the top-1000 items account for ~17.5% of all events. - Behavioral differences by context are pronounced: organic listens have higher skip rates (34.6% vs 26.0%) and lower full-listen rates (51.8% vs 63.2%) than non-organic listens.
- After sessionizing the event stream using a 30-minute inactivity threshold, sessions are typically short (median 4 events) but heavy-tailed (p99 ~71) with a median duration around 25 minutes.
- Sequential analysis shows sessions are mostly continuous listening (`listen→listen` ≈ 98.7%), with `listen→like` around 1.07%, and preference reversals are rare (`P(unlike|like)` ≈ 3.86%, appearing in ~0.62% of sessions). - Item-level sequentiality reveals replay behavior: 15.46% of sessions contain repeated tracks, and repetition is substantially more common in organic-majority sessions (repeat-containing sessions 21.4% vs 8.0%, and immediate repeats 2.87% vs 0.68%), suggesting organic use is more replay/exploitation-oriented while non-organic use is more linear/exploration-oriented.



In [1]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_parquet("hf://datasets/yandex/yambda/flat/50m/multi_event.parquet")

In [2]:
df.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,100,39420,8326270,0,100.0,170.0,listen
1,100,39420,1441281,0,100.0,105.0,listen
2,100,39625,286361,0,100.0,185.0,listen
3,100,40110,732449,0,100.0,240.0,listen
4,100,40360,3397170,0,46.0,130.0,listen


# 2. Research Question Definition

**RQ1: Segment users into behavioral types (explorers, replayers, curators)**

- Data mining task type: Clustering / representation learning

- Relevant algorithm(s)

  - Clustering: k-means / hierarchical clustering on user features

  - Graph Embeddings: learn user/item embeddings then cluster

  - Graph Mining: user degree / diversity / novelty features
  - Deep user embeddings (sequence encoders)

- Evaluation criteria: silhouette / Davies–Bouldin, stability across seeds, interpretability (feature differences), downstream usefulness (predict organic rate/skip)

**RQ2:Bot-like users detection**

- Data mining task type: Anomaly detection, stream regularity
Course technique(s):

- Relevant algorithm(s)
  - Anomaly Detection: outlier users by volume/repetition/regularity
  - Stream Mining: inter-event time regularity, bursts, periodicity
  - Graph Mining: suspicious user–item connectivity patterns

- Evaluation criteria: regularity metrics (gap variance/entropy), anomaly ranking plausibility, impact on popularity stats (top-K changes, concentration changes)

**RQ3:What is the causal effect of algorithmic (non-organic) exposure on skip probability, after adjusting for selection bias using propensity-score methods?**

- Data mining task type:Causal inference / treatment effect estimation (observational study; binary treatment is_organic=0 vs is_organic=1, outcome = skip indicator from played_ratio_pct)

- Relevant algorithm(s):

  - Propensity score estimation

  - Adjustment / effect estimation:IPW (Inverse Propensity Weighting),Matching (propensity score matching)

- Evaluation criteria :

  - ATE / ATT estimate (e.g., change in skip probability) + 95% confidence intervals

  - Covariate balance after adjustment (standardized mean differences; overlap/common support diagnostics)

  - Sensitivity checks (results stability across different feature sets, trimming extreme propensities, alternative skip thresholds)

Based on my EDA in stage one of this project, I don't need any more EDA to help me find interesting questions.

# 3. Motivation and Feasibility

## RQ1

In [3]:
import numpy as np
import pandas as pd

# convert to seconds since start
df["t_sec"] = df["timestamp"].astype("int64") * 5

# focus on listens for engagement + sequencing
listen = df[df["event_type"] == "listen"].copy()

# clean engagement view (drop missing engagement; flag invalid)
listen["played_invalid"] = listen["played_ratio_pct"].notna() & (
    (listen["played_ratio_pct"] < 0) | (listen["played_ratio_pct"] > 100)
)
listen["played_clip"] = listen["played_ratio_pct"].clip(0, 100)
listen["skip"] = listen["played_clip"] < 20
listen["full"] = listen["played_clip"] >= 95

# choose a manageable user subset
UIDS = listen["uid"].unique()
n_users = len(UIDS)
print("unique users in listen:", n_users)

# If runtime is slow, reduce:
sample_uids = np.random.choice(UIDS, size=min(3000, n_users), replace=False)
# sample_uids = UIDS  # use all available users by default

listen_s = listen[listen["uid"].isin(sample_uids)].copy()
df_s = df[df["uid"].isin(sample_uids)].copy()

# Sessionization (30 min inactivity) on listens
listen_s = listen_s.sort_values(["uid", "t_sec"])
gap = listen_s.groupby("uid")["t_sec"].diff().fillna(0)
listen_s["new_session"] = (gap > 1800).astype("int8")
listen_s["session_id"] = listen_s.groupby("uid")["new_session"].cumsum()

# session-level listen stats
sess = listen_s.groupby(["uid", "session_id"]).agg(
    n_listens=("item_id", "size"),
    n_unique=("item_id", "nunique"),
    dur_min=("t_sec", lambda x: (x.max() - x.min()) / 60.0),
    skip_rate=("skip", "mean"),
    full_rate=("full", "mean"),
    played_mean=("played_clip", "mean"),
)
sess["repeat_events"] = sess["n_listens"] - sess["n_unique"]
sess["has_repeat"] = sess["repeat_events"] > 0
sess["repeat_event_rate"] = np.where(sess["n_listens"] > 0, sess["repeat_events"] / sess["n_listens"], np.nan)

# immediate repeats (track == previous track) within session
listen_s["prev_item"] = listen_s.groupby(["uid", "session_id"])["item_id"].shift(1)
listen_s["imm_repeat"] = (listen_s["item_id"] == listen_s["prev_item"]).astype("int8")
imm_repeat_rate_sess = listen_s.groupby(["uid", "session_id"])["imm_repeat"].mean()

sess = sess.join(imm_repeat_rate_sess.rename("imm_repeat_rate"), how="left")

print("sessions:", len(sess))

# user-level aggregates from sessions
user_from_sess = sess.groupby("uid").agg(
    n_sessions=("n_listens", "size"),
    listens_per_session_med=("n_listens", "median"),
    dur_min_med=("dur_min", "median"),
    repeat_session_rate=("has_repeat", "mean"),
    repeat_event_rate_mean=("repeat_event_rate", "mean"),
    imm_repeat_rate_mean=("imm_repeat_rate", "mean"),
    skip_rate_mean=("skip_rate", "mean"),
    full_rate_mean=("full_rate", "mean"),
)

# user-level overall activity + feedback actions from all events
df_s = df_s.sort_values(["uid", "t_sec"])
user_activity = df_s.groupby("uid").agg(
    n_events=("event_type", "size"),
    n_items=("item_id", "nunique"),
    organic_share=("is_organic", "mean"),
    t_span_days=("t_sec", lambda x: (x.max() - x.min()) / 86400.0),
)

# event type rates per user (all events)
etype = pd.crosstab(df_s["uid"], df_s["event_type"], normalize="index")
etype = etype.add_prefix("rate_")

user_features = user_from_sess.join(user_activity, how="left").join(etype, how="left")

# derived: events/day and listens/day (avoid divide by 0)
user_features["events_per_day"] = user_features["n_events"] / (user_features["t_span_days"].clip(lower=1e-6))
# listens per day from listen_s
listens_per_user = listen_s.groupby("uid").size().rename("n_listens_total")
user_features = user_features.join(listens_per_user, how="left")
user_features["listens_per_day"] = user_features["n_listens_total"] / (user_features["t_span_days"].clip(lower=1e-6))

print("user_features shape:", user_features.shape)
display(user_features.head())

# quick feasibility checks (missingness + skew)
display(user_features.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))
display(user_features.describe(percentiles=[.5, .9, .95, .99]).T.head(20))


unique users in listen: 9238
sessions: 1634603
user_features shape: (3000, 20)


,n_sessions,listens_per_session_med,dur_min_med,repeat_session_rate,repeat_event_rate_mean,imm_repeat_rate_mean,skip_rate_mean,full_rate_mean,n_events,n_items,organic_share,t_span_days,rate_listen,rate_dislike,rate_like,rate_undislike,rate_unlike,events_per_day,n_listens_total,listens_per_day
uid,,,,,,,,,,,,,,,,,,,,
300,92,4.5,25.208333,0.076087,0.012185,0.006599,0.390880,0.431121,764,395,0.989529,1496.130498,0.921466,0.000000,0.057592,0.000000,0.020942,0.510651,704,0.470547
600,348,5.0,68.541667,0.002874,0.001437,0.001437,0.039839,0.855880,4549,1271,0.002198,1427.566551,1.000000,0.000000,0.000000,0.000000,0.000000,3.186541,4549,3.186541
1100,1263,3.0,25.000000,0.022961,0.006625,0.004112,0.109685,0.822537,4711,1849,0.143282,1180.671296,0.984292,0.000425,0.013585,0.000000,0.001698,3.990103,4637,3.927427
1900,1250,3.0,0.000000,0.056800,0.012720,0.006185,0.085682,0.808001,6718,3255,0.338494,1490.301505,0.947901,0.013695,0.034683,0.000298,0.003424,4.507813,6368,4.272961
2100,105,12.0,0.416667,0.247619,0.042704,0.011175,0.142355,0.746652,1433,838,0.865318,928.044850,0.979763,0.000000,0.018144,0.000000,0.002094,1.544106,1404,1.512858


,missing_rate
n_sessions,0.0
listens_per_session_med,0.0
dur_min_med,0.0
repeat_session_rate,0.0
repeat_event_rate_mean,0.0
imm_repeat_rate_mean,0.0
skip_rate_mean,0.0
full_rate_mean,0.0
n_events,0.0
n_items,0.0


,count,mean,std,min,50%,90%,95%,99%,max
n_sessions,3000.0,544.867667,604.079375,1.000000,327.000000,1399.200000,1834.150000,2648.230000,3853.000000
listens_per_session_med,3000.0,5.914000,4.186141,1.000000,5.000000,10.000000,13.000000,21.510000,67.000000
dur_min_med,3000.0,40.438403,47.812055,0.000000,30.000000,85.625000,115.010417,204.170833,987.916667
repeat_session_rate,3000.0,0.182530,0.148362,0.000000,0.148148,0.387032,0.474740,0.666667,1.000000
repeat_event_rate_mean,3000.0,0.045994,0.056170,0.000000,0.029216,0.101833,0.148048,0.270289,0.886364
imm_repeat_rate_mean,3000.0,0.021771,0.042562,0.000000,0.009803,0.047985,0.077783,0.214294,0.886364
skip_rate_mean,3000.0,0.256737,0.163406,0.000000,0.228720,0.469928,0.553141,0.799017,1.000000
full_rate_mean,3000.0,0.592264,0.208983,0.000000,0.610478,0.858823,0.905893,0.966801,1.000000
n_events,3000.0,5131.771000,5573.449276,11.000000,3143.500000,13356.200000,17459.200000,23975.660000,27699.000000
n_items,3000.0,1440.411000,1287.455802,2.000000,1092.000000,3203.100000,3953.800000,5741.890000,9963.000000


* **Motivation:** The aggregated user features show *large behavioral diversity*, not just noise. Users vary widely in **organic_share (0→1)**, **skip_rate_mean (0→1)**, **repeat_session_rate (0→1)**, and **median session size (1→67)**, suggesting multiple meaningful “usage styles” exist rather than one homogeneous population.

* **Non-triviality:** Simple global averages (or popularity-only views) would hide distinct patterns like **high-replay users** (high repeat rates), **high-skip samplers** (high skip, short sessions), and **curators** (higher like/unlike rates). The long-tailed distributions in activity and session behavior imply segmentation is needed to separate “power users” from genuine behavioral types.

* **Feasibility:** The dataset is suitable for clustering because based on the random 3000 user sample, now we have a complete user-feature matrix ** with no missingness** and strong variance across dimensions. Standard clustering (k-means / hierarchical) is computationally light at this scale, and the features are interpretable (session length, repeats, skip/full, organic share, like/unlike rates), making cluster labeling feasible.

* **Risks:**

  * **Skew/heavy tails** (e.g., `n_events`, `events_per_day`, `n_sessions`) can dominate distance-based methods unless we log-transform/standardize.
  * **Feature redundancy** (e.g., `n_events` vs `n_listens_total` vs rates) can create unstable clusters; pruning or PCA may be needed.
  * **Clusters may reflect activity level more than style** (power users vs casual), so we may need to normalize by time (`per_day`) and emphasize behavioral ratios.
  * **Choice of k** can be sensitive; we’ll need stability checks (silhouette/DB + multiple seeds).

## RQ2

In [4]:
# inter-event gaps on listens (per user)
gaps_sec = listen_s.groupby("uid")["t_sec"].diff().dropna()

# helper: compute per-user gap stats efficiently
gap_df = pd.DataFrame({
    "uid": gaps_sec.index.get_level_values(0) if isinstance(gaps_sec.index, pd.MultiIndex) else listen_s.loc[gaps_sec.index, "uid"].values,
    "gap_sec": gaps_sec.values
})

# When diff() is used on grouped series, index is original row index, so map uid via listen_s:
if "uid" not in gap_df or gap_df["uid"].isna().any():
    gap_df["uid"] = listen_s.loc[gaps_sec.index, "uid"].values

gap_stats = gap_df.groupby("uid").agg(
    gap_med=("gap_sec", "median"),
    gap_std=("gap_sec", "std"),
    gap_iqr=("gap_sec", lambda x: x.quantile(0.75) - x.quantile(0.25)),
    pct_gap_5s=("gap_sec", lambda x: (x == 5).mean()),
    pct_gap_10s=("gap_sec", lambda x: (x == 10).mean()),
    pct_gap_15s=("gap_sec", lambda x: (x == 15).mean()),
)

anomaly_table = user_features.join(gap_stats, how="left")

# simple anomaly scoring (lower variance + higher volume + higher exact-gap % + high repetition)
# (purely heuristic; good for feasibility EDA)
def z(x):
    return (x - x.mean()) / (x.std(ddof=0) + 1e-9)

anomaly_table["score_volume"] = z(np.log1p(anomaly_table["events_per_day"]))
anomaly_table["score_regular"] = -z(np.log1p(anomaly_table["gap_std"].fillna(anomaly_table["gap_std"].median())))
anomaly_table["score_exact"] = z((anomaly_table["pct_gap_5s"].fillna(0) + anomaly_table["pct_gap_10s"].fillna(0)))
anomaly_table["score_repeat"] = z(anomaly_table["imm_repeat_rate_mean"].fillna(0))

anomaly_table["anom_score"] = (
    0.4 * anomaly_table["score_volume"]
    + 0.3 * anomaly_table["score_regular"]
    + 0.2 * anomaly_table["score_exact"]
    + 0.1 * anomaly_table["score_repeat"]
)

print("\nTop 20 suspicious users by heuristic anomaly score:")
display(anomaly_table.sort_values("anom_score", ascending=False).head(20)[
    ["anom_score","events_per_day","listens_per_day","gap_med","gap_std","pct_gap_5s","pct_gap_10s",
     "imm_repeat_rate_mean","repeat_session_rate","organic_share","rate_listen","rate_like","rate_unlike"]
])


Top 20 suspicious users by heuristic anomaly score:


,anom_score,events_per_day,listens_per_day,gap_med,gap_std,pct_gap_5s,pct_gap_10s,imm_repeat_rate_mean,repeat_session_rate,organic_share,rate_listen,rate_like,rate_unlike
uid,,,,,,,,,,,,,
366300,1.951247,15.823758,15.016252,700.0,38619.313589,0.0,0.0,0.362143,0.742382,0.877489,0.948969,0.041785,0.008357
142400,1.907309,18.263520,17.641297,150.0,68149.090972,0.0,0.0,0.366878,0.719118,0.915019,0.965931,0.027689,0.005359
357200,1.669288,22.542555,21.681318,925.0,22498.078587,0.0,0.0,0.119539,0.330275,0.893420,0.961795,0.029563,0.007277
270200,1.634500,29.665725,28.092173,325.0,20481.362937,0.0,0.0,0.038979,0.463235,0.931469,0.946957,0.042283,0.010074
968100,1.624291,13.521482,13.002729,750.0,34171.016613,0.0,0.0,0.243422,0.521839,0.589268,0.961635,0.024574,0.012036
45300,1.588649,29.329688,29.288456,850.0,18045.422874,0.0,0.0,0.010105,0.312625,0.952671,0.998594,0.000703,0.000000
595900,1.524080,16.043132,15.903963,775.0,57919.904821,0.0,0.0,0.215148,0.591078,0.506988,0.991325,0.007711,0.000964
766100,1.508494,16.749237,16.357822,50.0,34409.331142,0.0,0.0,0.151472,0.474664,0.947190,0.976631,0.010661,0.010237
554500,1.506887,15.272542,15.004860,425.0,45679.498800,0.0,0.0,0.195804,0.459929,0.926884,0.982473,0.012818,0.003924


* **Motivation:** our anomaly table shows users with *highly unusual* interaction signatures—especially **extreme activity rates** (e.g., ~45 events/day for uid 839700) and **very high immediate repeat rates** (uid 200700 has `imm_repeat_rate_mean ≈ 0.505`, meaning ~50% of consecutive listens repeat the same track). And the median time between one listen and the next for this user is 0. These patterns are rare in typical human listening and are exactly the kinds of signals that can inflate item popularity and distort recommender evaluation.

* **Non-triviality:** Simple “heavy user” filtering is not enough. Our top-ranked users aren’t just high-volume; they also show **behavioral regularities** (very small median inter-event gaps like 25–175 seconds for some users, and unusually high repeat behavior) that suggest automation or abnormal usage modes. This requires anomaly detection on *temporal regularity + repetition structure*, not just counts.

* **Feasibility:** This method is practical because the dataset has the necessary observables: **timestamps (5s bins)** enable inter-event gap features (`gap_med`, `gap_std`), and sessionized listens enable **repeat signals** (`imm_repeat_rate_mean`, `repeat_session_rate`). Computing these features and ranking users is lightweight (10k users total), and the “impact” step is straightforward: recompute item popularity/top-K shares after removing flagged users and measure changes (e.g., top-1000 share shift, Jaccard overlap of top-K).

* **Risks:**

  * **False positives / ambiguity:** Background listening, shared devices, or power users can resemble “bot-like” behavior (e.g., long continuous listening with short gaps).
  * **Feature sensitivity:** Our heuristic score depends on thresholds/weights; different weights may reorder the top-20.
  * **Interpretation risk:** Some indicators here are not decisive—e.g., `pct_gap_5s` and `pct_gap_10s` are 0 for our top-20, so the strongest evidence is coming from **volume + gap regularity + repeats**, not exact 5s periodicity.
  * **Need for validation:** Without ground truth, we should validate via consistency checks (e.g., are these users disproportionately responsible for spikes or for a small set of items? do they have unusually concentrated item distributions?).

## RQ3

In [5]:
tmp = listen.copy()
tmp = tmp[tmp["track_length_seconds"].notna()].copy()
tmp["len_decile"] = pd.qcut(tmp["track_length_seconds"], 10, duplicates="drop")
len_overlap = tmp.groupby("len_decile")["is_organic"].agg(["mean","count"])
print("\nOrganic share by track-length decile:")
display(len_overlap)

# Use session_id built from listen_s (sampled users). For full listen, rebuild similarly.
pos = listen_s.copy()
pos["pos_in_sess"] = pos.groupby(["uid","session_id"]).cumcount() + 1
pos["pos_bucket"] = pd.cut(pos["pos_in_sess"], bins=[0,1,2,3,5,10,1e9], labels=["1","2","3","4-5","6-10","11+"])
pos_overlap = pos.groupby("pos_bucket")["is_organic"].agg(["mean","count"])
print("\nOrganic share by session position bucket:")
display(pos_overlap)

# skip rate by length decile and by position bucket (helps choose confounders)
tmp["skip"] = tmp["played_ratio_pct"].clip(0,100) < 20
skip_by_len = tmp.groupby("len_decile")["skip"].agg(["mean","count"])
print("\nSkip rate by track-length decile:")
display(skip_by_len)

pos2 = pos.copy()
pos2["skip"] = pos2["played_ratio_pct"].clip(0,100) < 20
skip_by_pos = pos2.groupby("pos_bucket")["skip"].agg(["mean","count"])
print("\nSkip rate by session position bucket:")
display(skip_by_pos)

/tmp/ipykernel_424/3164646624.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  len_overlap = tmp.groupby("len_decile")["is_organic"].agg(["mean","count"])



Organic share by track-length decile:


,mean,count
len_decile,,
"(4.999, 135.0]",0.480393,4791411
"(135.0, 160.0]",0.467219,5669577
"(160.0, 175.0]",0.484357,4688519
"(175.0, 185.0]",0.496554,3736982
"(185.0, 200.0]",0.506147,5604890
"(200.0, 210.0]",0.519136,3717242
"(210.0, 225.0]",0.531005,5212717
"(225.0, 240.0]",0.537891,3795433
"(240.0, 275.0]",0.558704,4903716


/tmp/ipykernel_424/3164646624.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pos_overlap = pos.groupby("pos_bucket")["is_organic"].agg(["mean","count"])



Organic share by session position bucket:


...


Skip rate by track-length decile:


,mean,count
len_decile,,
"(4.999, 135.0]",0.258198,4791411
"(135.0, 160.0]",0.276061,5669577
"(160.0, 175.0]",0.286139,4688519
"(175.0, 185.0]",0.293129,3736982
"(185.0, 200.0]",0.302506,5604890
"(200.0, 210.0]",0.308278,3717242
"(210.0, 225.0]",0.317810,5212717
"(225.0, 240.0]",0.323593,3795433
"(240.0, 275.0]",0.334310,4903716


/tmp/ipykernel_424/3164646624.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  skip_by_pos = pos2.groupby("pos_bucket")["skip"].agg(["mean","count"])



Skip rate by session position bucket:


,mean,count
pos_bucket,,
1,0.205045,1634603
2,0.205032,1329598
3,0.225363,1111319
4-5,0.254533,1737957
6-10,0.296036,2741029
11+,0.386136,6465551


* **Motivation:** Our overlap EDA shows **systematic selection** into organic vs non-organic contexts: organic share rises with **track length** (≈0.48 in shortest decile → ≈0.60 in longest) and **falls with session depth** (≈0.56 at position 1 → ≈0.48 at 11+). At the same time, skip rate **increases strongly** with both track length (≈0.26 → ≈0.36) and session depth (≈0.21 at pos 1–2 → ≈0.39 at 11+). This means the raw skip gap between organic and non-organic is likely confounded by *where in the session* the listen occurs and *how long the track is*—exactly the scenario where causal adjustment is motivated.

* **Non-triviality:** A naive comparison (skip|organic vs skip|non-organic) is not interpretable as a causal effect because **treatment assignment is not random**: algorithmic/non-organic listens occur more often later in sessions (where skips are much higher) and differ by track length distribution. Causal methods (propensity adjustment / doubly robust estimation) are needed to separate “exposure effect” from “context effect.”

* **Feasibility:** The key feasibility requirement—**common support/overlap**—looks good: every track-length decile and session-position bucket has **millions** of events in both contexts, and organic shares stay in a moderate range (roughly 0.47–0.60 by length; 0.48–0.56 by position), suggesting we won’t face severe separation. We also already have strong, measurable confounders available for a propensity model: `track_length_seconds` (or deciles) and `pos_in_session` (or buckets), plus optional user-level controls (baseline skip rate, activity level). This makes IPW/AIPW implementable and learnable in Colab on the 50M split (or on a large sample).

* **Risks:**

  * **Unobserved confounding:** even after controlling for length and session position, other factors (content quality, user intent, recommender surface) may still bias estimates.
  * **Model dependence:** results can vary with propensity model choice (logistic vs boosted trees) and trimming strategy.
  * **Positivity at finer granularity:** overlap is good at coarse bins, but could weaken if we condition on too many features (e.g., item-level).
  * **Computation:** full IPW/AIPW on tens of millions of rows can be heavy in pandas; we may need sampling or DuckDB/Polars for efficiency.


# 4. Methodological Planning

## RQ1: Segment users into behavioral types (explorers, replayers, curators)

* **Course algorithms:**

  * **k-means (W6)** — **Why:** scalable and interpretable on the full **user-feature table** (10k users). My EDA shows large variance across skip/repeat/organic/session features, so centroid-based clusters can capture distinct “typical” behavior modes.
  * **Hierarchical clustering, Ward (W6)** — **Why:** provides a structure check without relying on random initialization; Ward favors compact clusters, useful after standardization and for validating whether “few” clusters make sense.
  * **Graph Embeddings → cluster (W10)** — **Why:** complements feature-based clustering by capturing similarity from the **user–item interaction structure** (taste neighborhoods) that handcrafted features may miss.
  * **HDBSCAN** — **Why:** user features are heavy-tailed; density-based clustering can discover non-spherical clusters and label sparse regions as noise rather than forcing assignments.
* **External algorithms:**
  * **Gaussian Mixture Models (GMM)** — **Why:** supports **soft membership** (users can mix behaviors: replay + curation), which is realistic for listening behavior.
* **Evaluation (and why):**

  * **Silhouette, Davies–Bouldin** — **Why:** no ground truth labels; these quantify separation/compactness for method and k selection.
  * **Stability across seeds/subsamples** — **Why:** clustering can be sensitive; stability prevents over-interpreting artifacts.
  * **Interpretability via feature profiles** — **Why:** the end goal is human-labeled clusters (explorer/replayer/curator), not just numeric groupings.
* **Baselines (and why):**

  * **Rule-based thresholds** — **Why:** transparent benchmark; shows whether clustering adds value beyond simple heuristics.
  * **k-means on 2–3 features** — **Why:** tests whether segmentation is just “activity level” rather than meaningful multi-feature structure.

**Full-dataset execution note:** compute `user_features` from full 50M logs (aggregations + sessionized listen stats), then cluster on the 10k-row table.* **HDBSCAN** — **Why:** user features are heavy-tailed; density-based clustering can discover non-spherical clusters and label sparse regions as noise rather than forcing assignments.

## RQ2: Bot-like users detection (regular timing/extreme volume) + impact on popularity

* **Course algorithms:**

  * **Z-score/IQR outliers (W9)** — **Why:** explainable, audit-friendly anomaly ranking on key suspicious signals (extreme volume, abnormal regularity, repetition). Works well with heavy-tailed behavior seen in EDA.
  * **Stream-regularity features (W12)** — **Why:** bot-like behavior often manifests as unusual temporal structure (consistent pacing, bursts). Yambda’s 5s-binned timestamps enable gap-based regularity signals.
  * **Graph concentration signals (W3)** — **Why:** boosting/spam typically concentrates activity onto a small set of items; bipartite concentration (user→top items) distinguishes “power users” from “boosters.”
* **External algorithms:**

  * **Isolation Forest** — **Why:** captures multivariate outliers without manual thresholds; good robustness check when anomaly patterns are nonlinear.
  * **LOF** — **Why:** detects outliers relative to local neighborhoods (useful if there are multiple normal user regimes).
* **Evaluation (and why):**

  * **Overlap across detectors** — **Why:** without labels, agreement strengthens confidence; disagreement signals sensitivity.
  * **Manual plausibility on top-K** — **Why:** qualitative validation is necessary when ground truth is absent.
  * **Impact metrics (Top-K Jaccard, Δ top-100/1000 share, % events from flagged users)** — **Why:** directly tests whether anomalies distort the popularity distribution and evaluation-relevant head items.
* **Baselines (and why):**

  * **High-volume-only** — **Why:** simplest strawman; shows timing/repeat features add value beyond “power user = bot.”
  * **High-repeat-only** — **Why:** isolates looping behavior; compares single-symptom detection vs multi-signal detection.

**Full-dataset execution note:** compute anomaly features for all users from full logs; run detectors on the 10k-user table; recompute popularity before/after removing flagged users.

## RQ3: Causal effect of non-organic exposure on skip probability (propensity adjustment)

* **Course algorithms:**

  * **Logistic-regression propensity model (W5)** — **Why:** interpretable and stable baseline; scalable to large samples; produces calibrated propensities needed for weighting.
  * **Session-position features (W12)** — **Why:** My EDA shows strong confounding: organic share decreases with session depth while skip increases sharply with depth; ignoring this invalidates naive comparisons.
  * **Distributed / efficient aggregation (W8)** — **Why:** applying weights and computing weighted outcomes over tens of millions of rows requires scalable execution (DuckDB/Spark-style).
* **External algorithms:**

  * **IPW** — **Why:** corrects selection bias by reweighting observations to approximate randomized exposure.
  * **AIPW / Doubly Robust** — **Why:** more reliable: consistent if either propensity or outcome model is correct; reduces model dependence.
  * **PSM** — **Why:** provides an intuitive matched comparison as a check against weighting-based estimates.
* **Evaluation (and why):**

  * **ATE/ATT + 95% CI** — **Why:** causal question requires an effect size with uncertainty.
  * **Overlap/common support diagnostics** — **Why:** validity requires overlap; My binned EDA suggests overlap is reasonable but must be documented (and may require trimming).
  * **Balance checks (SMD)** — **Why:** shows whether adjustment actually removed measured confounding (track length, session position).
  * **Sensitivity tests (thresholds, trimming, model variants)** — **Why:** causal estimates can be sensitive; robustness increases credibility.
* **Baselines (and why):**

  * **Unadjusted skip difference** — **Why:** demonstrates confounding magnitude (how misleading raw gap is).
  * **Stratified comparison (length deciles × session position)** — **Why:** transparent “controlled” baseline that adjusts for the two strongest confounders My EDA identified without full propensity modeling.

**Full-dataset execution note:** train propensity/outcome models on a large stratified sample; apply IPW/AIPW via efficient aggregation over the full dataset; report diagnostics and sensitivity.


### RQ1 feasibility dry-runs

In [6]:

# Test: DuckDB can read the HF parquet and run a simple aggregation.
!pip -q install duckdb polars scikit-learn hdbscan statsmodels
import duckdb, pandas as pd

con = duckdb.connect()
con.execute("""
  SELECT event_type, COUNT(*) AS n
  FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
  GROUP BY event_type
""").df().head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,event_type,n
0,like,881456
1,listen,46467212
2,dislike,107776
3,undislike,21033
4,unlike,312972


In [9]:
# sample 200 users (fast)
uids = con.execute("""
  SELECT DISTINCT uid
  FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
  USING SAMPLE 0.05 PERCENT
  LIMIT 200
""").df()["uid"].tolist()

# pull listens for those users only
listen_small = con.execute(f"""
  SELECT uid, timestamp*5 AS t_sec, item_id, is_organic, played_ratio_pct, track_length_seconds, event_type
  FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
  WHERE uid IN ({",".join(map(str,uids))}) AND event_type='listen'
""").df()

listen_small.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(197448, 7)

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import numpy as np

# dummy feature example (replace with your engineered columns once computed)
X = listen_small[["t_sec"]].values  # placeholder to confirm pipeline
X = StandardScaler().fit_transform(X)

def fit_kmeans(X, k):
    return MiniBatchKMeans(n_clusters=k, n_init=5, batch_size=2048, random_state=0).fit_predict(X)

for k in range(2, 5):
    labels = fit_kmeans(X, k)
    sil = silhouette_score(X, labels, sample_size=2000, random_state=0)
    print(k, sil)

2 0.622536081962869
3 0.5967743718543829
4 0.5577405247000108


Confirms sklearn works, scaling/standardization pipeline is correct.

### RQ2 feasibility dry-runs

Use DuckDB to compute per-user event counts and listen counts (no sessionization yet). Confirms full-user aggregation is feasible and fast.

In [22]:
user_counts = con.execute("""
  SELECT uid,
         COUNT(*) AS n_events,
         SUM(CASE WHEN event_type='listen' THEN 1 ELSE 0 END) AS n_listens,
         AVG(is_organic) AS organic_share
  FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
  GROUP BY uid
""").df()

user_counts.shape, user_counts.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

((10000, 4),
      uid  n_events  n_listens  organic_share
 0  60900      3589     3501.0       0.787127
 1  61200        50        0.0       1.000000
 2  64000      1382     1376.0       0.637482
 3  64100     18695    17880.0       0.932228
 4  73900      5885     5848.0       0.193713)

Isolation Forest on a few numeric features (n_events, n_listens, organic_share).Confirms package works and produces a stable ranking.

In [23]:
from sklearn.ensemble import IsolationForest

X = user_counts[["n_events","n_listens","organic_share"]].values
iso = IsolationForest(random_state=0, n_estimators=200, contamination=0.01)
scores = -iso.fit(X).score_samples(X)
user_counts["iso_score"] = scores
user_counts.sort_values("iso_score", ascending=False).head(10)

,uid,n_events,n_listens,organic_share,iso_score
2404,650600,26381,26342.0,0.016489,0.754171
6011,725800,26019,26019.0,0.000000,0.752132
8538,246000,26025,26013.0,0.000384,0.751878
4342,997500,27699,27496.0,0.062999,0.751370
976,596900,25660,25660.0,0.000000,0.750100
5609,702000,27040,26560.0,0.976886,0.749592
9187,668400,25726,25704.0,0.036656,0.749592
6727,509900,27578,27178.0,0.932084,0.748579
9395,646100,27387,27154.0,0.077957,0.748325
1110,83700,27624,27617.0,0.119498,0.747061


Remove top-20 suspicious users and recompute top-100 item share (on a sample).Confirms my removal + recomputation pipeline works.

In [24]:
flagged = set(user_counts.sort_values("iso_score", ascending=False).head(20)["uid"])

impact = con.execute(f"""
  WITH base AS (
    SELECT item_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
    WHERE event_type='listen'
    GROUP BY item_id
  ),
  filtered AS (
    SELECT item_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
    WHERE event_type='listen' AND uid NOT IN ({",".join(map(str, flagged))})
    GROUP BY item_id
  )
  SELECT
    (SELECT SUM(c) FROM (SELECT c FROM base ORDER BY c DESC LIMIT 100))::DOUBLE / (SELECT SUM(c) FROM base) AS top100_share_base,
    (SELECT SUM(c) FROM (SELECT c FROM filtered ORDER BY c DESC LIMIT 100))::DOUBLE / (SELECT SUM(c) FROM filtered) AS top100_share_filtered
""").df()

impact

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,top100_share_base,top100_share_filtered
0,0.042174,0.042403


### RQ3 feasibility dry-runs

Pull ~1–5M listens using DuckDB sampling, with:

- treatment T = 1-is_organic (non-organic)

- outcome Y = played_ratio_pct < 20 (with clipping)

- covariates: track length + session position proxy (see note)

Why: Confirms we can assemble a training dataset and the columns behave.

In [26]:
sample = con.execute("""
  SELECT
    uid,
    timestamp*5 AS t_sec,
    is_organic,
    CASE WHEN played_ratio_pct < 20 THEN 1 ELSE 0 END AS skip,
    track_length_seconds
  FROM read_parquet('hf://datasets/yandex/yambda/flat/50m/multi_event.parquet')
  WHERE event_type='listen' AND played_ratio_pct IS NOT NULL AND track_length_seconds IS NOT NULL
  USING SAMPLE 2 PERCENT
""").df()

sample.shape, sample.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

((902507, 5),
     uid     t_sec  is_organic  skip  track_length_seconds
 0  1900  13585750           1     0                   170
 1  1900  13585750           1     0                   260
 2  1900  13585750           1     0                   265
 3  1900  13585750           1     0                   250
 4  1900  13585750           1     0                   220)

Logistic regression propensity on track_length_seconds only (dry-run), compute IPW ATE. Ensures my causal stack runs end-to-end.

In [27]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# Treatment: non-organic exposure
T = (sample["is_organic"] == 0).astype(int).values
Y = sample["skip"].astype(int).values

X = sample[["track_length_seconds"]].values

prop = LogisticRegression(max_iter=200)
prop.fit(X, T)
ps = prop.predict_proba(X)[:,1]

# trim to avoid crazy weights in dry run
eps = 1e-3
ps = np.clip(ps, eps, 1-eps)

w = T/ps + (1-T)/(1-ps)
ate_ipw = np.average(Y*T/ps) / np.average(T/ps) - np.average(Y*(1-T)/(1-ps)) / np.average((1-T)/(1-ps))
ate_ipw

np.float64(-0.07960295302078946)

On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- https://huggingface.co/datasets/yandex/yambda?library=pandas

3. AI Tools:
- ChatGPT: I gave it my previous EDA notebook and my RQs, and aksed it to generate some tests.